# Employetics — `AdOne` (40s Kinetic Ad) Colab Renderer

Renders **`AdOne`** (`1920x1080`, `30 fps`, `1200 frames / 40.0 s`, H.264 `yuv420p`, CRF 18) cleanly on Google Colab.

- Handles **Colab root Chromium sandbox** via `remotion.config.ts` (`--no-sandbox`, `--disable-setuid-sandbox`, `--disable-dev-shm-usage`).
- Supports both **Ubuntu 24.04 (`libasound2t64`)** and **Ubuntu 22.04 (`libasound2`)** per Remotion's Linux dependencies docs.

In [ ]:
# Cell 1: Install Node.js 20 (nodesource) + Remotion Linux Chrome dependencies
%%bash
set -e

echo "=== 1) Installing Node.js 20 (NodeSource) ==="
curl -fsSL https://deb.nodesource.com/setup_20.x | bash -
apt-get install -y nodejs unzip
node -v && npm -v

echo "=== 2) Installing Remotion Linux shared library dependencies ==="
apt-get update -qq
ASOUND_PKG="libasound2t64"
if ! apt-cache show libasound2t64 >/dev/null 2>&1; then
  ASOUND_PKG="libasound2"
fi
echo "Using audio package: ${ASOUND_PKG}"

apt-get install -y -qq \
  libnss3 libdbus-1-3 libatk1.0-0 "${ASOUND_PKG}" \
  libxrandr2 libxkbcommon-dev libxfixes3 libxcomposite1 \
  libxdamage1 libatk-bridge2.0-0 libpango-1.0-0 libcairo2 \
  libcups2 libgbm-dev

echo "=== 3) Installing edge-tts for optional voice-over generation ==="
pip install -q edge-tts
echo "Cell 1 complete."

In [ ]:
# Cell 2: Upload & unzip project (or clone from GitHub), npm install, ensure Remotion browser
import os
import subprocess
from pathlib import Path

WORK_DIR = Path("/content/employeticsAD")
USE_GITHUB = False  # Set True to clone directly from GitHub instead of uploading a zip
GENERATE_VOICEOVER = True  # Set False to render SFX-only / custom audio/voiceover.mp3

if USE_GITHUB:
    if WORK_DIR.exists():
        subprocess.run(["git", "-C", str(WORK_DIR), "pull", "origin", "main"], check=True)
    else:
        subprocess.run(["git", "clone", "https://github.com/abdullahmehmood1/EmployeticsAD.git", str(WORK_DIR)], check=True)
else:
    from google.colab import files
    print("Upload employetics_ad_colab.zip (created by scripts/package_for_colab.sh):")
    uploaded = files.upload()
    zip_name = next(iter(uploaded.keys()))
    subprocess.run(["rm", "-rf", str(WORK_DIR)], check=True)
    WORK_DIR.mkdir(parents=True, exist_ok=True)
    subprocess.run(["unzip", "-qo", zip_name, "-d", str(WORK_DIR)], check=True)

os.chdir(WORK_DIR)
print("Working directory:", os.getcwd())

# Install npm dependencies & ensure Remotion headless browser
subprocess.run(["npm", "install"], check=True)
subprocess.run(["npx", "remotion", "browser", "ensure"], check=True)

# Generate procedural SFX if missing
if not (WORK_DIR / "public" / "sfx" / "impact.wav").exists():
    subprocess.run(["python3", "scripts/make_sfx.py"], check=True)

# Optionally generate edge-tts voiceover clips unless custom audio/voiceover.mp3 is supplied
if GENERATE_VOICEOVER and not (WORK_DIR / "audio" / "voiceover.mp3").exists():
    subprocess.run(["python3", "scripts/make_voiceover.py"], check=True)

subprocess.run(["node", "scripts/sync_assets.mjs"], check=True)
print("Cell 2 complete — ready to test render!")

In [ ]:
# Cell 3: Quick test render of frames 0-60 at scale 0.5 to verify Chromium & pipeline
%%bash
set -e
cd /content/employeticsAD
mkdir -p out
npx remotion render src/index.ts AdOne out/test_0_60.mp4 \
  --frames=0-60 \
  --scale=0.5 \
  --codec=h264 \
  --crf=18 \
  --pixel-format=yuv420p \
  --gl=swangle \
  --concurrency=2
ls -lh out/test_0_60.mp4
echo "Quick test render PASSED!"

In [ ]:
# Cell 4: Full 1080p 30fps 40.0s (1200 frames) render -> out/ad1.mp4
%%bash
set -e
cd /content/employeticsAD
mkdir -p out
npx remotion render src/index.ts AdOne out/ad1.mp4 \
  --codec=h264 \
  --crf=18 \
  --pixel-format=yuv420p \
  --gl=swangle \
  --concurrency=2

echo "=== Output Verification ==="
ls -lh out/ad1.mp4
if command -v ffprobe >/dev/null 2>&1; then
  ffprobe -v error -select_streams v:0 \
    -show_entries stream=codec_name,width,height,r_frame_rate,nb_frames,duration \
    -of default=noprint_wrappers=1 out/ad1.mp4
fi

In [ ]:
# Cell 5: Download out/ad1.mp4
from google.colab import files
files.download("/content/employeticsAD/out/ad1.mp4")